In [0]:
import logging

logger = logging.getLogger("bronze-ingestion")
logger.setLevel(logging.INFO)

jobParameters = dbutils.widgets.getAll()

fileName = jobParameters.get("file_name", "").strip()
storageAccountName = jobParameters.get("storage_account_name", "").strip()
sourceContainerName = jobParameters.get("source_container", "").strip()
targetContainerName = jobParameters.get("target_container", "").strip()

if not fileName:
    raise ValueError(
        f"Missing required parameter: file_name. "
        f"Received parameters: {jobParameters}"
    )

if not storageAccountName:
    raise ValueError(
        f"Missing required parameter: storage_account_name. "
        f"Received parameters: {jobParameters}"
    )

if not sourceContainerName:
    raise ValueError(
        f"Missing required parameter: source_container. "
        f"Received parameters: {jobParameters}"
    )

if not targetContainerName:
    raise ValueError(
        f"Missing required parameter: target_container. "
        f"Received parameters: {jobParameters}"
    )

sourcePath = (
    f"abfss://{sourceContainerName}@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

schemaLocation = (
    f"abfss://{targetContainerName}@{storageAccountName}.dfs.core.windows.net/schemas/{fileName}"
)

checkpointLocation = (
    f"abfss://{targetContainerName}@{storageAccountName}.dfs.core.windows.net/checkpoints/{fileName}"
)

outputPath = (
    f"abfss://{targetContainerName}@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

logger.info("Starting bronze ingestion for file: %s", fileName)
logger.info("Source path: %s", sourcePath)
logger.info("Output path: %s", outputPath)
logger.info("Schema location: %s", schemaLocation)
logger.info("Checkpoint location: %s", checkpointLocation)

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schemaLocation)
    .load(sourcePath)
)

query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpointLocation)
    .option("path", outputPath)
    .trigger(once=True)
    .start()
)

query.awaitTermination()

logger.info("Bronze ingestion completed for file: %s", fileName)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8570130615001402>, line 14
     11 targetContainerName = jobParameters.get("target_container", "").strip()
     13 if not fileName:
---> 14     raise ValueError(
     15         f"Missing required parameter: file_name. "
     16         f"Received parameters: {jobParameters}"
     17     )
     19 if not storageAccountName:
     20     raise ValueError(
     21         f"Missing required parameter: storage_account_name. "
     22         f"Received parameters: {jobParameters}"
     23     )

ValueError: Missing required parameter: file_name. Received parameters: {'target_container': '', 'table_names': '', 'file_name': '', 'storage_account_name': '', 'source_container': ''}